In [49]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor

from sklearn.metrics import mean_absolute_error

In [50]:
train = pd.read_csv('churn_train.csv')
test = pd.read_csv('churn_test.csv')
print("train :", train.shape)
print("test :", test.shape)

train : (4116, 19)
test : (1764, 18)


In [51]:
target = train.pop('TotalCharges')

In [52]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4116 entries, 0 to 4115
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   customerID        4116 non-null   object
 1   gender            4116 non-null   object
 2   SeniorCitizen     4116 non-null   int64 
 3   Partner           4116 non-null   object
 4   Dependents        4116 non-null   object
 5   tenure            4116 non-null   int64 
 6   PhoneService      4116 non-null   object
 7   MultipleLines     4116 non-null   object
 8   InternetService   4116 non-null   object
 9   OnlineSecurity    4116 non-null   object
 10  OnlineBackup      4116 non-null   object
 11  DeviceProtection  4116 non-null   object
 12  TechSupport       4116 non-null   object
 13  StreamingTV       4116 non-null   object
 14  StreamingMovies   4116 non-null   object
 15  Contract          4116 non-null   object
 16  PaperlessBilling  4116 non-null   object
 17  PaymentMethod 

In [53]:
train.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
dtype: int64

In [54]:
##id값 제거
train = train.drop('customerID', axis=1)
test = test.drop(['customerID'], axis=1)

In [55]:
cols = train.select_dtypes('object').columns
cols

Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object')

In [56]:
encoder = LabelEncoder()
for col in cols:
    train[col] = encoder.fit_transform(train[col])
    test[col] = encoder.transform(test[col])

train.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod
0,1,0,0,0,7,1,2,2,1,1,1,1,1,1,2,0,2
1,0,1,0,1,53,0,1,1,2,2,0,2,2,2,0,0,1
2,0,1,0,1,68,0,1,0,2,2,2,0,0,2,0,0,3
3,1,1,0,1,44,0,1,2,1,1,1,1,1,1,0,1,1
4,1,0,1,0,7,1,2,1,0,2,2,0,2,0,1,0,2


In [63]:
x_train, x_val, y_train, y_val = train_test_split(train, target, test_size=0.2, random_state=0)

print("x_train :", x_train.shape)
print("x_val :", x_val.shape)
print("y_train :", y_train.shape)
print("y_val :", y_val.shape)

x_train : (3292, 17)
x_val : (824, 17)
y_train : (3292,)
y_val : (824,)


In [64]:
model1 = RandomForestRegressor(random_state=0)
model1.fit(x_train,y_train)
pred = model1.predict(x_val)

In [65]:
mae = mean_absolute_error(y_val, pred)
print("랜덤포레스트 :", mae)

랜덤포레스트 : 951.0960435538027


In [66]:
model2 = LGBMRegressor(random_state=0, verbose=-1)
model2.fit(x_train,y_train)
pred = model2.predict(x_val)

In [67]:
mae = mean_absolute_error(y_val,pred)
print("LGBM :", mae)

LGBM : 952.7925407798712


In [68]:
pred = model1.predict(test)
result = pd.DataFrame({
    'pred' : pred
})
result.to_csv('result.csv', index=False)

In [69]:
result.head()

,pred
0,3707.6212
1,923.7132
2,4057.4078
3,952.6143
4,1322.1638
